# FinGPT → MedicalGPT：最小 SFT 训练流程（Notebook）

目标：实现一个**最小可跑**的 SFT 流程：
1. 加载 FinGPT 数据（参考 FinGPT 的 HF dataset 用法）
2. 清洗/转换成本项目 SFT 数据格式（参考 `docs/datasets.md`）
3. 运行 SFT（参考 `supervised_finetuning.py` 与 `run_training_dpo_pipeline.ipynb`）

数据格式要求（SFT）：jsonl，每行一个样本，包含 `conversations` 字段，且每轮为：
- `{ "from": "human", "value": "..." }`
- `{ "from": "gpt", "value": "..." }`


## 0. 环境准备（可选）

如果你是在全新环境运行，建议先安装依赖（已配置好环境可跳过）。


## 1. 配置参数（最小可跑）

默认选择小模型（便于快速验证）。如需更大模型/更多 steps，自行调整。

数据集选择：`fingpt-sentiment-train`
 - 任务最直观：情感分类（正/中/负）比 NER、关系抽取、复杂问答更容易理解与调参。
 - 官方训练路径最成熟：FinGPT 文档里“task-specific”和“multi-task”都把 sentiment-train 放在核心位置，说明这条链路最稳。​
 - 社区复现/对比最多：FinGPT v3 系列主要围绕情感任务做了完整 benchmark 和训练说明，新手更容易找到可参考结果。​

 如果你只是想快速冒烟（几十分钟内看通路），可以先用更小数据如 fingpt-ner（样本少），但任务本身更“结构化抽取”，对新手反而不一定更容易。

 如果你想做中文金融选择题风格，再考虑 fingpt-fineval。


In [1]:
# HF Mirror 数据源（解决国内访问 Hugging Face 慢的问题）
HF_ENDPOINT = "https://hf-mirror.com"


In [2]:
# 设置环境变量（优先走 HF Mirror）
import os
os.environ["HF_ENDPOINT"] = HF_ENDPOINT


In [3]:
# 先确认环境变量已正确加载
print("HF_ENDPOINT 环境变量值:", os.getenv("HF_ENDPOINT"))  # 应输出 https://hf-mirror.com

HF_ENDPOINT 环境变量值: https://hf-mirror.com


In [4]:
from pathlib import Path

# ====== 可改参数 ======
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct" # https://hf-mirror.com/Qwen/Qwen2.5-7B-Instruct
FIN_DATASET = "FinGPT/fingpt-sentiment-train" 
FIN_SPLIT = "train"

OUT_DIR = Path("data/fingpt_min")
RAW_DIR = OUT_DIR / "raw"
SFT_DIR = OUT_DIR / "sft"

RAW_FILE = RAW_DIR / "fingpt_raw.jsonl"
SFT_FILE = SFT_DIR / "fingpt_sft_sharegpt.jsonl"

SFT_OUT = Path("outputs/fingpt_sft_lora")
MERGED_OUT = Path("outputs/fingpt_sft_merged")  # 可选

RAW_DIR.mkdir(parents=True, exist_ok=True)
SFT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_MODEL:", BASE_MODEL)
print("FIN_DATASET:", FIN_DATASET, "split=", FIN_SPLIT)
print("RAW_FILE:", RAW_FILE)
print("SFT_FILE:", SFT_FILE)
print("SFT_OUT:", SFT_OUT)


BASE_MODEL: Qwen/Qwen2.5-7B-Instruct
FIN_DATASET: FinGPT/fingpt-sentiment-train split= train
RAW_FILE: data/fingpt_min/raw/fingpt_raw.jsonl
SFT_FILE: data/fingpt_min/sft/fingpt_sft_sharegpt.jsonl
SFT_OUT: outputs/fingpt_sft_lora


## 2. 加载 FinGPT 数据并保存为本地 jsonl（参考 FinGPT 用法）

这里使用 HuggingFace `datasets.load_dataset` 直接加载 `FinGPT/...` 数据集。


In [5]:
from huggingface_hub import HfApi
print(HfApi().endpoint)
# 示例：自定义Hugging Face端点
hf_api = HfApi(endpoint="https://hf-mirror.com")
print("自定义端点:", hf_api.endpoint)  # 输出 https://hf-mirror.com

https://hf-mirror.com
自定义端点: https://hf-mirror.com


In [7]:
import json
from datasets import load_dataset
# 参考 FinGPT：直接从 HF datasets 加载 FinGPT 数据集
ds = load_dataset(FIN_DATASET, split=FIN_SPLIT)
with RAW_FILE.open("w", encoding="utf-8") as f:
    for row in ds:
        f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

print("saved rows:", len(ds))
print("saved to:", RAW_FILE)


(…)-00000-of-00001-dabab110260ac909.parquet:   0%|          | 0.00/6.42M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/76772 [00:00<?, ? examples/s]

saved rows: 76772
saved to: data/fingpt_min/raw/fingpt_raw.jsonl


## 3. 清洗/转换为 SFT 格式（符合 `docs/datasets.md`）

本项目已提供 `fin_to_sharegpt.py`，会把 FinGPT 风格字段映射为 ShareGPT `conversations` 格式（`human/gpt` 两轮）。


In [8]:
import subprocess

subprocess.run(
    [
        "python",
        "fin_to_sharegpt.py",
        "--source_file",
        str(RAW_FILE),
        "--output_file",
        str(SFT_FILE),
    ],
    check=True,
)
print("converted ->", SFT_FILE)


Generating train split: 76772 examples [00:00, 129188.16 examples/s]


Saved 76772 SFT rows to data/fingpt_min/sft/fingpt_sft_sharegpt.jsonl
converted -> data/fingpt_min/sft/fingpt_sft_sharegpt.jsonl


## 4. 运行 SFT（LoRA）（参考 `supervised_finetuning.py`）

这里用极小的训练步数（`--max_steps 50`）做 smoke test，确保全流程可跑通。

如果你需要完整训练：
- 去掉 `--max_steps` 改用 `--num_train_epochs` + 更大 batch/accumulation
- 或者设置 `--max_train_samples -1` 跑全量


In [ ]:
!python supervised_finetuning.py \
    --model_name_or_path BASE_MODEL \
    --tokenizer_name_or_path BASE_MODEL \
    --train_file_dir SFT_DIR \
    --validation_split_percentage 1 \
    --do_train \
    --use_peft True \
    --num_train_epochs 1 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 2 \
    --learning_rate 2e-4 \
    --max_steps 50 \
    --logging_steps 5 \
    --save_steps 50 \
    --model_max_length 512 \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype float16 \
    --device_map auto \
    --output_dir SFT_OUT \
    --overwrite_output_dir


## 5.（可选）合并 LoRA 权重到 base model

如果你希望导出一个可直接推理的合并模型，可以执行合并。


In [ ]:
import subprocess

# 可选：合并 LoRA 方便推理部署
cmd = [
    "python",
    "merge_peft_adapter.py",
    "--base_model",
    BASE_MODEL,
    "--tokenizer_path",
    BASE_MODEL,
    "--lora_model",
    str(SFT_OUT),
    "--output_dir",
    str(MERGED_OUT),
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)
